In [5]:
!pip install -q ultralytics roboflow torch

In [6]:
import torch

def selecionar_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

device = selecionar_device()
device

device(type='cpu')

In [7]:
import os, numpy as np
from PIL import Image

ALVO = 512
path_images_fold = "Tumor-Cerebral-1"    # a mesma pasta, será alterada


EXTENSOES_IMAGEM = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")

def padronizar(im):
    im = im.convert("RGB")
    w, h = im.size
    a = np.asarray(im)
    borda = np.concatenate([a[0], a[-1], a[:,0], a[:,-1]])
    fundo = tuple(int(x) for x in np.median(borda, axis=0))
    s = ALVO / max(w, h)
    nw, nh = max(1, round(w*s)), max(1, round(h*s))
    im = im.resize((nw, nh), Image.LANCZOS)
    tela = Image.new("RGB", (ALVO, ALVO), fundo)
    tela.paste(im, ((ALVO-nw)//2, (ALVO-nh)//2))
    return tela

processadas, puladas, ignoradas = 0, 0, 0

for r, _, fs in os.walk(path_images_fold):
    for f in fs:
        # ignora .txt e qualquer arquivo que não seja imagem
        if not f.lower().endswith(EXTENSOES_IMAGEM):
            ignoradas += 1
            continue

        origem = os.path.join(r, f)
        destino = os.path.splitext(origem)[0] + ".jpg"
        im = Image.open(origem)

        # pula o que já está pronto
        if im.size == (ALVO, ALVO) and im.mode == "RGB" and origem == destino:
            puladas += 1
            continue

        nova = padronizar(im)
        im.close()
        nova.save(destino, quality=95)
        if origem != destino:
            os.remove(origem)
        processadas += 1

print(f"Processadas: {processadas} | Já prontas: {puladas} | Ignoradas (não são imagem): {ignoradas}")

Processadas: 0 | Já prontas: 6879 | Ignoradas (não são imagem): 4


In [8]:
from ultralytics import YOLO

model = YOLO("yolo11s-cls.pt")

results = model.train(
  data=path_images_fold,
  epochs=100,
  imgsz=ALVO,
  batch=32,
  optimizer="AdamW",
  lr0=0.0003,
  cos_lr=True,
  patience=30,
  dropout=0.3,
  freeze=10,
  erasing=0.1,
  seed=42,
  auto_augment=None,
  hsv_h=0.0, hsv_s=0.0, hsv_v=0.2,   # só um pouco de brilho/contraste
  scale=0.3, translate=0.05,
  fliplr=0.5, flipud=0.0,
  weight_decay=0.01,
  project="runs", name="yolo11s_lung_v2",
)


New https://pypi.org/project/ultralytics/8.4.199 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.194  Python-3.14.0 torch-2.11.0+cpu CPU (Intel Core Ultra 5 235U)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=None, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=Tumor-Cerebral-1, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.3, dynamic=False, embed=None, epochs=100, erasing=0.1, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.0, hsv_s=0.0, hsv_v=0.2, imgsz=512, iou=0.7, kobj=1.0, line_width=None, lr0=0.0003, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s-cls.pt, momentum=0.937, mosaic=1.0, m

In [9]:
m_val  = model.val(data=path_images_fold, split="val",  imgsz=ALVO)
m_test = model.val(data=path_images_fold, split="test", imgsz=ALVO)
print("valid top1:", m_val.top1, "| test top1:", m_test.top1)

Ultralytics 8.4.194  Python-3.14.0 torch-2.11.0+cpu CPU (Intel Core Ultra 5 235U)
YOLO11s-cls summary (fused): 47 layers, 5,439,252 parameters, 0 gradients, 12.1 GFLOPs
train: C:\Users\pedro.almeida\Downloads\Trabalho Faculdade\Tumor-Cerebral-1\train... found 4819 images in 4 classes  
val: C:\Users\pedro.almeida\Downloads\Trabalho Faculdade\Tumor-Cerebral-1\valid... found 1379 images in 4 classes  
test: C:\Users\pedro.almeida\Downloads\Trabalho Faculdade\Tumor-Cerebral-1\test... found 681 images in 4 classes  
val: Fast image access  (ping: 0.1±0.0 ms, read: 60.2±33.5 MB/s, size: 32.0 KB)
val: Scanning C:\Users\pedro.almeida\Downloads\Trabalho Faculdade\Tumor-Cerebral-1\valid... 1379 images, 0 corrupt: 100% ━━━━━━━━━━━━ 1379/1379 304.4Mit/s 0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 87/87 1.0s/it 1:311.0sss
                   all      0.971          1
Speed: 0.0ms preprocess, 57.9ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to C:\Users